In [8]:
"""
Sort a shipping/route Excel sheet and flag reverse-direction duplicates.

Sort order:
    1. Origin State
    2. Origin City
    3. Destination State
    4. Destination City

Duplicate check:
    Flags any pair of rows where Origin -> Destination in one row matches
    Destination -> Origin in another row (e.g. "Atlanta, GA -> Macon, GA"
    and "Macon, GA -> Atlanta, GA" would both be flagged).

Usage:
    Just set INPUT_PATH and OUTPUT_PATH below, then run:
    python sort_and_check_routes.py
"""

import pandas as pd


# ---- SET YOUR FILE PATHS HERE ----
INPUT_PATH = r"C:\Users\KyleSteen.AzureAD\Documents\Task 2.1\Corridors_10_2.xlsx"
OUTPUT_PATH = r"C:\Users\KyleSteen.AzureAD\Documents\Task 2.1\Corridors_10_2_UPDATED.xlsx"
# -----------------------------------

REQUIRED_COLS = ["City A", "State A", "City B", "State B"]


def load_data(input_path: str) -> pd.DataFrame:
    df = pd.read_excel(input_path)

    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected column(s): {missing}")

    return df


def sort_routes(df: pd.DataFrame) -> pd.DataFrame:
    """Sort alphabetically: Origin State -> Origin City -> Destination State -> Destination City."""
    sort_cols = ["State A", "City A", "State B", "City B"]

    sorted_df = df.sort_values(
        by=sort_cols,
        key=lambda col: col.astype(str).str.strip().str.lower(),
        kind="mergesort",  # stable sort, keeps original relative order for exact ties
    ).reset_index(drop=True)

    return sorted_df


def find_reverse_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Find rows whose route (Origin -> Destination) is the exact reverse of
    another row's route (Destination -> Origin), e.g. Atlanta,GA->Macon,GA
    and Macon,GA->Atlanta,GA.

    Returns a DataFrame of the flagged rows, with a 'Duplicate Group' column
    linking matched pairs/sets together. Empty DataFrame if none found.
    """
    work = df.copy()

    # Normalize for comparison (case/whitespace-insensitive matching)
    work["_o_city"] = work["City A"].astype(str).str.strip().str.lower()
    work["_o_state"] = work["State A"].astype(str).str.strip().str.lower()
    work["_d_city"] = work["City B"].astype(str).str.strip().str.lower()
    work["_d_state"] = work["State B"].astype(str).str.strip().str.lower()

    # A "route key" that is the SAME regardless of direction:
    # sort the (city, state) endpoint pair so A->B and B->A produce identical keys.
    def route_key(row):
        origin = (row["_o_city"], row["_o_state"])
        dest = (row["_d_city"], row["_d_state"])
        return tuple(sorted([origin, dest]))

    work["_route_key"] = work.apply(route_key, axis=1)

    # Only keep groups where there's more than one row sharing that undirected key
    # AND the rows aren't literally identical direction (i.e. real A->B / B->A pairs,
    # not just duplicate rows of the same direction -- though those get flagged too
    # since they still share the same undirected key).
    dup_mask = work.groupby("_route_key")["_route_key"].transform("count") > 1
    dups = work[dup_mask].copy()

    if dups.empty:
        return df.iloc[0:0]  # empty, same columns

    # Give each duplicate group a readable ID, ordered by first appearance
    group_ids = {key: i + 1 for i, key in enumerate(dups["_route_key"].drop_duplicates())}
    dups["Duplicate Group"] = dups["_route_key"].map(group_ids)

    dups = dups.sort_values("Duplicate Group")
    dups = dups.drop(columns=["_o_city", "_o_state", "_d_city", "_d_state", "_route_key"])

    return dups


def main():
    input_path, output_path = INPUT_PATH, OUTPUT_PATH

    df = load_data(input_path)
    sorted_df = sort_routes(df)
    dups = find_reverse_duplicates(df)

    # Write sorted data + a duplicates sheet (if any) into the output workbook
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        sorted_df.to_excel(writer, sheet_name="Sorted Routes", index=False)
        if not dups.empty:
            dups.to_excel(writer, sheet_name="Possible Duplicates", index=False)

    print(f"Sorted {len(sorted_df)} rows -> '{output_path}' (sheet: Sorted Routes)")
    if dups.empty:
        print("No reverse-direction duplicate routes found.")
    else:
        n_groups = dups["Duplicate Group"].nunique()
        print(
            f"Found {len(dups)} rows across {n_groups} duplicate group(s) "
            f"-> see 'Possible Duplicates' sheet."
        )


if __name__ == "__main__":
    main()


Sorted 319 rows -> 'C:\Users\KyleSteen.AzureAD\Documents\Task 2.1\Corridors_10_2_UPDATED.xlsx' (sheet: Sorted Routes)
Found 4 rows across 2 duplicate group(s) -> see 'Possible Duplicates' sheet.
